# Alpha-beta pruning

_Artificial Intelligence (CS221)_

**Skip branches that cannot change your decision. Same answer, far less work.**

Minimax explores the whole game tree. That can be enormous.

     Alpha-beta pruning skips branches that cannot possibly affect the final choice.

---

This notebook builds **alpha-beta pruning** one piece at a time. Run each cell, read the note above it to see *why* each branch can be safely skipped, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — Python

### Step 1 — Lay out the game tree

We represent a tiny two-ply game tree as nested lists. The **root is a MAX node** (our move): it picks the child with the largest value. Each of its children is a **MIN node** (the opponent's move): it picks the child with the smallest value. The leaves are plain integers — the payoffs to MAX.

`visited` is a one-element list used as a mutable counter so the recursion can tally how many leaves it actually looks at. With pruning we expect to examine *fewer* than all 8 leaves.

In [ ]:
tree = [[[3, 12], [8, 2]], [[4, 6], [14, 5]]]
visited = [0]                            # mutable counter of leaves examined

### Step 2 — Search with alpha and beta windows

`alpha` is the best score MAX can already guarantee so far; `beta` is the best (lowest) score MIN can already guarantee. Together they form a *window* of values still worth exploring.

At a MAX node we raise `alpha` as we find better children; the moment `alpha >= beta`, MIN above us would never let us reach this branch, so we **prune** the remaining children (a *beta cutoff*). At a MIN node the mirror image happens: we lower `beta`, and once `alpha >= beta` we prune (an *alpha cutoff*). A leaf just returns its value and bumps the counter.

In [ ]:
def ab(node, alpha, beta, maximizing):
    # A leaf is just an integer payoff — record it and return.
    if isinstance(node, int):
        visited[0] += 1
        return node

    if maximizing:
        v = -10**9                       # MAX starts from the worst possible value
        for child in node:
            child_value = ab(child, alpha, beta, False)
            v = max(v, child_value)
            alpha = max(alpha, v)        # MAX can now guarantee at least v
            if alpha >= beta:
                break                    # beta cutoff: prune the rest
        return v
    else:
        v = 10**9                        # MIN starts from the best possible value
        for child in node:
            child_value = ab(child, alpha, beta, True)
            v = min(v, child_value)
            beta = min(beta, v)          # MIN can now hold MAX to at most v
            if alpha >= beta:
                break                    # alpha cutoff: prune the rest
        return v

### Step 3 — Run it and count the savings

We launch the search at the root as a MAX node, with the widest possible window (`alpha = -∞`, `beta = +∞`). The returned value is the optimal score MAX can force. Because of the cutoffs, the leaf counter will read **less than 8** — those skipped leaves are branches alpha-beta proved could not change the answer.

In [ ]:
value = ab(tree, -10**9, 10**9, True)
print("alpha-beta value:", value)
print("leaves examined :", visited[0], "of 8 total (rest were pruned)")

## Visualize the data & results

_On that same tic-tac-toe position, how many game positions does alpha-beta skip versus full minimax?_

### Step 1 — A win-checker and a mid-game board

Here we move from an abstract tree to a real game. `winner(b)` scans the eight winning lines of a tic-tac-toe board and returns `"X"`, `"O"`, or `None`. The `board` is a partially-played position with X to move; we will search the rest of the game from here two different ways and compare how many positions each one visits. `counts` accumulates those tallies.

In [ ]:
def winner(b):
    lines = [(0,1,2),(3,4,5),(6,7,8),(0,3,6),(1,4,7),(2,5,8),(0,4,8),(2,4,6)]
    for a, c, d in lines:
        if b[a] != " " and b[a] == b[c] == b[d]:
            return b[a]
    return None

board = ["O","O"," "," ","X"," "," "," "," "]
counts = {"full": 0, "ab": 0}

### Step 2 — Full minimax vs alpha-beta on the same position

`full` walks **every** legal continuation, incrementing `counts["full"]` at each position — this is plain minimax with no pruning. `ab` evaluates the same game (X is MAX, O is MIN; a win for X scores `+1`, a win for O scores `-1`, a draw `0`) but carries `alpha`/`beta` and breaks out the moment `alpha >= beta`. Each visited position bumps `counts["ab"]`, so the gap between the two counters is exactly what pruning saved.

In [ ]:
def full(b, player):
    counts["full"] += 1
    if winner(b) or " " not in b:
        return
    for i in range(9):
        if b[i] == " ":
            next_player = "O" if player == "X" else "X"
            full(b[:i] + [player] + b[i+1:], next_player)

def ab(b, player, alpha, beta):
    counts["ab"] += 1
    w = winner(b)
    if w == "X":
        return 1
    if w == "O":
        return -1
    if " " not in b:
        return 0

    if player == "X":                    # X is the maximizing player
        v = -2
        for i in range(9):
            if b[i] == " ":
                child = ab(b[:i] + ["X"] + b[i+1:], "O", alpha, beta)
                v = max(v, child)
                alpha = max(alpha, v)
                if alpha >= beta:
                    break
        return v

    v = 2                                # O is the minimizing player
    for i in range(9):
        if b[i] == " ":
            child = ab(b[:i] + ["O"] + b[i+1:], "X", alpha, beta)
            v = min(v, child)
            beta = min(beta, v)
            if alpha >= beta:
                break
    return v

### Step 3 — Plot positions examined

We run both searches from the same board, then draw a bar for each. The alpha-beta bar should be visibly shorter: it reaches the *same* optimal decision while expanding far fewer positions, which is the whole point of pruning.

In [ ]:
full(board, "X")
ab(board, "X", -2, 2)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["alpha-beta", "full minimax"], [counts["ab"], counts["full"]],
       color=["#7ee787", "#ffb454"])
ax.text(0, counts["ab"], str(counts["ab"]), ha="center", va="bottom")
ax.text(1, counts["full"], str(counts["full"]), ha="center", va="bottom")
ax.set_title("positions examined: alpha-beta vs minimax")
plt.show()